# Pipeline RLHF complet — Qwen2.5-1.5B sur ETHICS

**Avant de lancer :** `Runtime → Change runtime type → T4 GPU`

Ensuite : `Runtime → Run all`

Durée estimée sur T4 : ~3-4h (reward model ~1h, RLOO ~2h, éval ~30min)

## 0. Setup

In [ ]:
!nvidia-smi | grep -E 'Tesla|T4|A100|L4|V100|Driver' || echo '❌ PAS DE GPU — Runtime → Change runtime type → T4'

In [ ]:
%cd /content
!rm -rf repo
!git clone --quiet --branch feat/Reinforcement_Learning_from_Human_Feedback \
    https://github.com/Crams0n/Project-Ethical-Alignment-of-Small-Language-Models.git repo
!pip install -q -r /content/repo/requirements.txt
print('✅ Repo cloné + dépendances installées')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/rlhf_outputs', exist_ok=True)
!rm -rf /content/repo/outputs
!ln -sf '/content/drive/MyDrive/rlhf_outputs' /content/repo/outputs
print('✅ Drive monté — outputs/ → MyDrive/rlhf_outputs/')

In [ ]:
import os, sys, json, gc
from pathlib import Path
import torch
import numpy as np

REPO = Path('/content/repo')
sys.path.insert(0, str(REPO))
os.chdir(REPO)

from src.utils.config import load_config
from src.utils.seed import seed_everything
from src.utils.device import runtime_profile, adapt_quant_cfg

profile = runtime_profile()
cfg     = load_config('configs/config.yaml')
seed_everything(cfg['seed'])
cfg['quantization'] = adapt_quant_cfg(cfg['quantization'], profile)

print('device :', profile)
print('GPU    :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('base   :', cfg['base_model'])

---
## 1. Reward Model Training
Entraîne un reward model sur les paires HH-RLHF (chosen vs rejected) avec LoRA sur Qwen2.5-1.5B.  
**Output :** `outputs/reward_model/`

In [ ]:
from peft import TaskType
from trl import RewardConfig, RewardTrainer
from src.data.preferences import load_hh_rlhf_for_reward_model
from src.models.reward import load_base_reward_model, make_lora_config

train_ds, eval_ds = load_hh_rlhf_for_reward_model(
    num_train=cfg['reward_model']['num_train_samples'],
    num_eval=cfg['reward_model']['num_eval_samples'],
    seed=cfg['seed'],
)
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in ('chosen', 'rejected')])
eval_ds  = eval_ds.remove_columns([c for c in eval_ds.column_names  if c not in ('chosen', 'rejected')])
print(f'train: {len(train_ds)}  eval: {len(eval_ds)}')

In [ ]:
model, tokenizer = load_base_reward_model(
    model_name=cfg['base_model'],
    quant_cfg=cfg['quantization'],
)
peft_config = make_lora_config(cfg['lora'], task_type=TaskType.SEQ_CLS)

rm_cfg     = cfg['reward_model']
output_dir = cfg['paths']['reward_model_dir']
Path(output_dir).mkdir(parents=True, exist_ok=True)

reward_config = RewardConfig(
    output_dir=output_dir,
    num_train_epochs=rm_cfg['num_train_epochs'],
    per_device_train_batch_size=rm_cfg['per_device_train_batch_size'],
    per_device_eval_batch_size=rm_cfg['per_device_train_batch_size'],
    gradient_accumulation_steps=rm_cfg['gradient_accumulation_steps'],
    learning_rate=rm_cfg['learning_rate'],
    warmup_ratio=rm_cfg['warmup_ratio'],
    logging_steps=rm_cfg['logging_steps'],
    eval_strategy='steps',
    eval_steps=rm_cfg['eval_steps'],
    save_steps=rm_cfg['save_steps'],
    save_total_limit=2,
    bf16=profile.use_bf16,
    gradient_checkpointing=profile.has_cuda,
    max_length=rm_cfg['max_length'],
    report_to='none',
    seed=cfg['seed'],
)

rm_trainer = RewardTrainer(
    model=model,
    args=reward_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)
rm_result = rm_trainer.train()
print(rm_result.metrics)

In [ ]:
rm_trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
metrics = rm_trainer.evaluate()
with open(Path(output_dir) / 'final_metrics.json', 'w') as fh:
    json.dump(metrics, fh, indent=2)
print('✅ Reward model sauvegardé dans', output_dir)

# Libérer la VRAM avant RLOO
del rm_trainer, model, tokenizer, train_ds, eval_ds
gc.collect(); torch.cuda.empty_cache()

---
## 2. RLOO Policy Optimization
Fine-tune la politique avec REINFORCE Leave-One-Out contre le reward model.  
**Output :** `outputs/rloo_policy/`

In [ ]:
from transformers import AutoTokenizer
from trl import RLOOConfig, RLOOTrainer
from src.utils.prompts import format_chat_for_generation
from src.data.preferences import load_hh_rlhf_prompts_for_rl
from src.models.reward import load_base_causal_lm, load_reward_model_for_inference

_tmp_tok = AutoTokenizer.from_pretrained(cfg['base_model'])
prompts_ds = load_hh_rlhf_prompts_for_rl(
    num_prompts=cfg['rloo']['num_prompts'],
    seed=cfg['seed'],
)
max_chars = cfg['rloo']['max_prompt_length'] * 4
prompts_ds = prompts_ds.map(lambda x: {'prompt': format_chat_for_generation(_tmp_tok, x['prompt'])})
prompts_ds = prompts_ds.filter(lambda r: len(r['prompt']) < max_chars)
del _tmp_tok
print(f'{len(prompts_ds)} prompts')

In [ ]:
policy, tokenizer = load_base_causal_lm(
    model_name=cfg['base_model'],
    quant_cfg=cfg['quantization'],
)
policy_peft_config = make_lora_config(cfg['lora'], task_type=TaskType.CAUSAL_LM)

reward_model, reward_tokenizer = load_reward_model_for_inference(
    base_model_name=cfg['base_model'],
    adapter_path=cfg['paths']['reward_model_dir'],
    quant_cfg=cfg['quantization'],
)
reward_model.eval()
for p in reward_model.parameters():
    p.requires_grad_(False)

rl_cfg  = cfg['rloo']
out_dir = cfg['paths']['rloo_policy_dir']
Path(out_dir).mkdir(parents=True, exist_ok=True)

rloo_config = RLOOConfig(
    output_dir=out_dir,
    num_train_epochs=rl_cfg['num_train_epochs'],
    per_device_train_batch_size=rl_cfg['per_device_train_batch_size'],
    gradient_accumulation_steps=rl_cfg['gradient_accumulation_steps'],
    learning_rate=rl_cfg['learning_rate'],
    warmup_ratio=rl_cfg['warmup_ratio'],
    logging_steps=rl_cfg['logging_steps'],
    save_steps=rl_cfg['save_steps'],
    save_total_limit=2,
    bf16=profile.use_bf16,
    gradient_checkpointing=profile.has_cuda,
    num_generations=rl_cfg['num_generations'],
    max_completion_length=rl_cfg['max_completion_length'],
    beta=rl_cfg['beta'],
    temperature=rl_cfg['temperature'],
    top_p=rl_cfg['top_p'],
    top_k=rl_cfg['top_k'],
    log_completions=True,
    num_completions_to_print=2,
    report_to='none',
    seed=cfg['seed'],
)

rloo_trainer = RLOOTrainer(
    model=policy,
    reward_funcs=reward_model,
    args=rloo_config,
    train_dataset=prompts_ds,
    processing_class=tokenizer,
    reward_processing_classes=reward_tokenizer,
    peft_config=policy_peft_config,
)
rloo_result = rloo_trainer.train()
print(rloo_result.metrics)

In [ ]:
rloo_trainer.save_model(out_dir)
tokenizer.save_pretrained(out_dir)
with open(Path(out_dir) / 'final_metrics.json', 'w') as fh:
    json.dump(rloo_result.metrics, fh, indent=2)
print('✅ Politique RLOO sauvegardée dans', out_dir)

del rloo_trainer, policy, reward_model, tokenizer, reward_tokenizer, prompts_ds
gc.collect(); torch.cuda.empty_cache()

---
## 3. Évaluation sur ETHICS
Compare baseline vs modèle aligné sur 5 sous-ensembles ETHICS.  
**Output :** `outputs/eval/results.json` + `comparison.png`

In [ ]:
from src.data.ethics import load_all_ethics
from src.evaluation.ethics_eval import evaluate_all, summarize, load_causal_lm_for_eval

eval_cfg = cfg['evaluation']
examples = load_all_ethics(
    subsets=tuple(eval_cfg['ethics_subsets']),
    num_samples=eval_cfg['samples_per_subset'],
    seed=cfg['seed'],
)
for name, exs in examples.items():
    print(f'{name:>16}: {len(exs)} exemples  (positive rate = {np.mean([e.label for e in exs]):.2f})')

In [ ]:
def evaluate_one(name, adapter_path):
    print(f'\n=== {name} (adapter={adapter_path}) ===')
    model, tok = load_causal_lm_for_eval(cfg['base_model'], adapter_path)
    res     = evaluate_all(model, tok, examples)
    summary = summarize(res)
    print(f'accuracies: {summary}')
    del model, tok
    gc.collect(); torch.cuda.empty_cache()
    return {'summary': summary, 'per_subset': {k: {'accuracy': v.accuracy, 'n': v.n} for k, v in res.items()}}

all_results = {}
for name, spec in eval_cfg['models'].items():
    all_results[name] = evaluate_one(name, spec.get('adapter_path'))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

eval_dir = Path(cfg['paths']['eval_dir'])
eval_dir.mkdir(parents=True, exist_ok=True)

with open(eval_dir / 'results.json', 'w') as fh:
    json.dump(all_results, fh, indent=2)

rows = []
for model_name, payload in all_results.items():
    for subset, stats in payload['per_subset'].items():
        rows.append({'model': model_name, 'subset': subset, 'accuracy': stats['accuracy']})

df = pd.DataFrame(rows).pivot(index='subset', columns='model', values='accuracy')
df.loc['average'] = df.mean()
df['delta'] = df.get('rlhf', 0) - df.get('baseline', 0)
df.to_csv(eval_dir / 'comparison.csv')
display(df)

plot_df = df.drop(index='average', errors='ignore').drop(columns='delta', errors='ignore')
ax = plot_df.plot(kind='bar', figsize=(9, 5), width=0.75)
ax.set_ylabel('accuracy')
ax.set_ylim(0, 1)
ax.set_title('ETHICS accuracy — baseline vs. RLHF-aligned (Qwen2.5-1.5B)')
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='chance')
ax.legend(loc='lower right')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(eval_dir / 'comparison.png', dpi=160)
plt.show()
print('✅ Résultats sauvegardés dans', eval_dir)